# Eau et agriculture : qualité des eaux et usage des pesticides

**Problématique :** _à formuler_

Auteurs : _à compléter_

## 0. Imports et configuration

In [ ]:
import os

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from src.hubeau.recuperer_toutes_stations import recuperer_toutes_stations
from src.hubeau.recuperer_tous_indices import recuperer_tous_indices
from src.geo.recuperer_departements import recuperer_departements
from src.nettoyage.nettoyer_indices import nettoyer_indices
from src.nettoyage.agreger_eau import agreger_eau
from src.visualisation.tracer_evolution_indices import tracer_evolution_indices
from src.visualisation.tracer_carte import tracer_carte
from src.visualisation.tracer_stations import tracer_stations

plt.rcParams.update({
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e5e5e5",
})

## 1. Récupération des données

1.1 Stations de mesure des cours d'eau (Hub'Eau)  
1.2 Indices biologiques de qualité des cours d'eau (Hub'Eau)  
1.3 Ventes de pesticides (BNV-D)  
1.4 Données agricoles (RPG)  
1.5 Fonds de carte

### 1.1 Stations de mesure des cours d'eau

Stations des 96 départements de métropole.

In [ ]:
CHEMIN_STATIONS = "data/raw/stations_rivieres.parquet"

if os.path.exists(CHEMIN_STATIONS):
    stations = pd.read_parquet(CHEMIN_STATIONS)
else:
    stations = recuperer_toutes_stations()
    stations.to_parquet(CHEMIN_STATIONS, index=False)

stations.shape

### 1.2 Indices biologiques

La qualité d'une rivière se mesure aussi par les êtres vivants qu'on y trouve. Trois indices, calculés sur 2015-2022 :

| Indice | Ce qu'il observe | Échelle | Sensible à |
|---|---|---|---|
| **I2M2** | invertébrés (larves, vers, mollusques) | 0 à 1, **haut = bonne qualité** | pollutions multiples, dont les pesticides |
| **IBD** | diatomées (micro-algues) | 0 à 20, **haut = bonne qualité** | nutriments (engrais) et matière organique |
| **IPR** | poissons | 0 et plus, **bas = bonne qualité** | état physique de la rivière, pollutions |

In [ ]:
CHEMIN_INDICES = "data/raw/indices_hydrobio.parquet"

if os.path.exists(CHEMIN_INDICES):
    indices = pd.read_parquet(CHEMIN_INDICES)
else:
    indices = recuperer_tous_indices()
    indices.to_parquet(CHEMIN_INDICES, index=False)

indices.shape

### 1.5 Fonds de carte

Contours simplifiés des départements ([france-geojson](https://github.com/gregoiredavid/france-geojson)), projetés en Lambert 93 (EPSG:2154), la projection officielle de la métropole.

In [ ]:
CHEMIN_DEPARTEMENTS = "data/geo/departements.parquet"

if os.path.exists(CHEMIN_DEPARTEMENTS):
    departements = gpd.read_parquet(CHEMIN_DEPARTEMENTS)
else:
    departements = recuperer_departements()
    departements.to_parquet(CHEMIN_DEPARTEMENTS)

departements = departements.to_crs("EPSG:2154")

## 2. Nettoyage et mise en forme

### 2.1 Qualité des cours d'eau : `eau.parquet`

On garde les indices qualifiés « corrects », on retire les valeurs codées 999 de l'IPR et les doublons, puis on agrège au département × année × indice.

In [ ]:
indices_propres = nettoyer_indices(indices)
eau = agreger_eau(indices_propres)
eau.to_parquet("data/processed/eau.parquet", index=False)

print("doublons sur la clé :", eau.duplicated(subset=["code_dep", "annee", "indice"]).sum())
eau.shape

## 3. Statistiques descriptives

### 3.1 Couverture des données

Nombre de stations suivies par année, pour chaque indice.

In [ ]:
indices_propres.groupby(["annee", "indice"])["code_station"].nunique().unstack()

Chaque année, environ 3 000 à 5 000 stations ont un I2M2 ou un IBD, et 1 500 à 2 200 un IPR. Ce ne sont pas toujours les mêmes stations d'une année à l'autre : une partie du réseau n'est suivie qu'une année sur deux ou trois.

### 3.2 Distribution des indices (2015-2022)

In [ ]:
indices_propres.groupby("indice")["valeur"].describe().round(2)

Les trois indices sont très dispersés d'une station à l'autre : l'I2M2 couvre toute l'échelle de 0 à 1. L'IPR a quelques valeurs extrêmes (jusqu'à 165) : on s'appuie donc sur la **médiane** plutôt que sur la moyenne.

### 3.3 Évolution 2015-2022

Valeur médiane par année. L'axe de l'IPR est inversé : sur les trois panneaux, **le haut signifie une meilleure qualité**.

In [ ]:
tracer_evolution_indices(indices_propres);

Les variations sont faibles (attention, les axes ne partent pas de 0) : la médiane de l'I2M2 reste entre 0,52 et 0,60, celle de l'IBD entre 15,1 et 15,5. L'I2M2 et l'IPR se dégradent légèrement sur la période ; l'IBD est stable. Les changements de stations suivies d'une année à l'autre peuvent suffire à créer ces écarts.

## 4. Visualisations et cartes

### 4.1 Stations suivies

Stations ayant au moins un indice biologique entre 2015 et 2022.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
tracer_stations(stations[stations["code_station"].isin(indices_propres["code_station"])], departements, ax)
ax.set_title("Stations avec un indice biologique (2015-2022)");

Le réseau couvre tout le territoire. Il est particulièrement dense en Auvergne-Rhône-Alpes.

### 4.2 Qualité biologique par département

Valeur médiane de chaque indice sur 2015-2022. La palette de l'IPR est inversée : sur les trois cartes, **foncé = meilleure qualité**. Chaque carte a sa propre échelle.

In [ ]:
medianes = indices_propres.groupby(["code_dep", "indice"])["valeur"].median().unstack().reset_index()

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
tracer_carte(departements, medianes, "I2M2", "I2M2 (invertébrés)", axes[0])
tracer_carte(departements, medianes, "IBD", "IBD (diatomées)", axes[1])
tracer_carte(departements, medianes, "IPR", "IPR (poissons)", axes[2], cmap="Blues_r")

Les trois indices dessinent une géographie proche :
- la qualité est **plus faible** dans les grandes plaines agricoles : Bassin parisien, Nord, Centre-Ouest (Vendée, Charentes, Deux-Sèvres) ;
- elle est **meilleure** dans les massifs (Massif central, Pyrénées, Alpes) et en Corse, où l'agriculture est moins intensive.

C'est cette géographie qu'il faudra confronter aux ventes de pesticides (BNV-D) et aux surfaces cultivées (RPG). Attention : l'altitude, la pente et la taille des cours d'eau jouent aussi sur ces indices, indépendamment de l'agriculture. L'IPR n'est pas disponible en Corse (en gris).

## 5. Croisement eau × agriculture

## 6. Conclusion et limites